In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os
zip_path = '/content/drive/MyDrive/first.aid.zip'
extract_path = '/content/dataset'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("تم فك الضغط بنجاح!")

تم فك الضغط بنجاح!


In [ ]:

if os.path.exists('/content/dataset'):
    print("المحتويات الموجودة جوه dataset هي:")
    print(os.listdir('/content/dataset'))
else:
    print("مجلد dataset مش موجود أصلاً، ارجعي لخطوة فك الضغط (Unzip)")

المحتويات الموجودة جوه dataset هي:
['first.aid']


In [ ]:
import os

base_path = '/content/dataset/first.aid'

if os.path.exists(base_path):
    print("المجلدات الأساسية:", os.listdir(base_path))

    train_path = os.path.join(base_path, 'train')
    if os.path.exists(train_path):
        print("الفئات جوه train:", os.listdir(train_path))
else:
    print("المسار لسه فيه مشكلة، تأكدي من الحروف)")

المجلدات الأساسية: ['val', 'train', 'test']
الفئات جوه train: ['abrasion', 'laceration', 'normal skin', 'bruise', 'burn', 'cut']


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils import shuffle, class_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# =========================
# 📌 1. DATA PREPROCESSING
# =========================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_dir = '/content/dataset/first.aid/train'
val_dir = '/content/dataset/first.aid/val'

datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True
)

train_gen = datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_gen = datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =========================
# 📌 2. CLASS BALANCING
# =========================
weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)
class_weights_dict = dict(enumerate(weights))

# =========================
# 📌 3. MODEL BUILDING
# =========================
def build_model():
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )

    for layer in base_model.layers[:-30]:
        layer.trainable = False

    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs)

    # CNN Block
    x = layers.Conv2D(256, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    # Transformer Block
    s = x.shape
    x = layers.Reshape((s[1]*s[2], s[3]))(x)

    attention = layers.MultiHeadAttention(num_heads=8, key_dim=64)(x, x)
    x = layers.Add()([x, attention])
    x = layers.LayerNormalization()(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    return models.Model(inputs, x)

extractor = build_model()

classifier = models.Sequential([
    extractor,
    layers.Dense(6, activation='softmax')
])

# =========================
# 📌 4. CALLBACKS
# =========================
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,   # عشان ما يوقفش قبل 13
    restore_best_weights=True
)

lr_scheduler = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2,
    min_lr=1e-7
)

# =========================
# 📌 5. COMPILE
# =========================
classifier.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# =========================
# 📌 6. TRAINING (13 Epoch)
# =========================
classifier.fit(
    train_gen,
    validation_data=val_gen,
    epochs=13,   # 👈 هنا التعديل
    class_weight=class_weights_dict,
    callbacks=[early_stop, lr_scheduler]
)

# =========================
# 📌 7. SAVE MODEL
# =========================
classifier.save('/content/model.h5')

# =========================
# 📌 8. FEATURE EXTRACTION
# =========================
train_gen_fixed = datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

X_train = extractor.predict(train_gen_fixed)
X_val = extractor.predict(val_gen)

y_train = train_gen_fixed.classes
y_val = val_gen.classes

X_train, y_train = shuffle(X_train, y_train)

# =========================
# 📌 9. ENSEMBLE
# =========================
xgb = XGBClassifier(n_estimators=1200, learning_rate=0.01)
lgbm = LGBMClassifier(n_estimators=1200, learning_rate=0.01)
rf = RandomForestClassifier(n_estimators=600)

ensemble = VotingClassifier(
    estimators=[('xgb', xgb), ('lgbm', lgbm), ('rf', rf)],
    voting='soft'
)

ensemble.fit(X_train, y_train)

# =========================
# 📌 10. EVALUATION
# =========================
preds = ensemble.predict(X_val)

print("Final Accuracy:", accuracy_score(y_val, preds))
print(classification_report(y_val, preds))

Found 9946 images belonging to 6 classes.
Found 866 images belonging to 6 classes.
Epoch 1/13
311/311 ━━━━━━━━━━━━━━━━━━━━ 215s 589ms/step - accuracy: 0.5059 - loss: 1.3117 - val_accuracy: 0.7552 - val_loss: 0.7902 - learning_rate: 1.0000e-05
Epoch 2/13
311/311 ━━━━━━━━━━━━━━━━━━━━ 156s 502ms/step - accuracy: 0.7463 - loss: 0.7730 - val_accuracy: 0.8418 - val_loss: 0.4994 - learning_rate: 1.0000e-05
Epoch 3/13
311/311 ━━━━━━━━━━━━━━━━━━━━ 154s 495ms/step - accuracy: 0.8120 - loss: 0.5628 - val_accuracy: 0.8822 - val_loss: 0.3740 - learning_rate: 1.0000e-05
Epoch 4/13
311/311 ━━━━━━━━━━━━━━━━━━━━ 158s 508ms/step - accuracy: 0.8479 - loss: 0.4489 - val_accuracy: 0.9030 - val_loss: 0.3181 - learning_rate: 1.0000e-05
Epoch 5/13
311/311 ━━━━━━━━━━━━━━━━━━━━ 155s 497ms/step - accuracy: 0.8753 - loss: 0.3793 - val_accuracy: 0.9273 - val_loss: 0.2555 - learning_rate: 1.0000e-05
Epoch 6/13
311/311 ━━━━━━━━━━━━━━━━━━━━ 163s 523ms/step - accuracy: 0.9008 - loss: 0.3126 - val_accuracy: 0.9261 - va

Found 9946 images belonging to 6 classes.
311/311 ━━━━━━━━━━━━━━━━━━━━ 150s 454ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 17s 601ms/step
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.132365 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 130560
[LightGBM] [Info] Number of data points in the train set: 9946, number of used features: 512
[LightGBM] [Info] Start training from score -1.787345
[LightGBM] [Info] Start training from score -1.769488
[LightGBM] [Info] Start training from score -1.803084
[LightGBM] [Info] Start training from score -1.809818
[LightGBM] [Info] Start training from score -1.801255
[LightGBM] [Info] Start training from score -1.780164
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Final Accuracy: 0.9491916859122402
              precision    recall  f1-score   support

           0       0.95      0.98      0.96       151
           1       0.97      0.93      0.95       257
           2       0.87      0.96      0.91        94
           3       0.89      0.98      0.93        81
           4       0.97      0.87      0.92        95
           5       0.99      0.97      0.98       188

    accuracy                           0.95       866
   macro avg       0.94      0.95      0.94       866
weighted avg       0.95      0.95      0.95       866



In [ ]:
# =========================
# 📊 1. TRAINING ACCURACY (FIXED)
# =========================
# نعمل generator بدون shuffle عشان الترتيب يبقى صحيح
train_gen_fixed = datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    shuffle=False
)

# توقعات الموديل
train_preds = classifier.predict(train_gen_fixed)
train_preds = np.argmax(train_preds, axis=1)

# حساب الدقة
train_acc = accuracy_score(train_gen_fixed.classes, train_preds)

print(f"🧠 Training Accuracy: {train_acc * 100:.2f}%")


# =========================
# 📊 2. VALIDATION ACCURACY (DL MODEL)
# =========================
val_preds_dl = classifier.predict(val_gen)
val_preds_dl = np.argmax(val_preds_dl, axis=1)

val_acc_dl = accuracy_score(val_gen.classes, val_preds_dl)

print(f"📊 Validation Accuracy (DL Model): {val_acc_dl * 100:.2f}%")


# =========================
# 📊 3. VALIDATION ACCURACY (ENSEMBLE)
# =========================
val_preds_ens = ensemble.predict(X_val)

val_acc_ens = accuracy_score(y_val, val_preds_ens)

print(f"🔥 Validation Accuracy (Ensemble): {val_acc_ens * 100:.2f}%")

Found 9946 images belonging to 6 classes.
311/311 ━━━━━━━━━━━━━━━━━━━━ 140s 451ms/step
🧠 Training Accuracy: 97.75%
28/28 ━━━━━━━━━━━━━━━━━━━━ 12s 444ms/step
📊 Validation Accuracy (DL Model): 95.73%


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


🔥 Validation Accuracy (Ensemble): 94.92%


In [ ]:
import numpy as np
import cv2
import tensorflow as tf

# =========================
# 📌 1. CLASS NAMES
# =========================
class_labels = list(train_gen.class_indices.keys())


# =========================
# 📌 2. SEVERITY LOGIC
# =========================
def estimate_severity(injury, confidence):

    severe_cases = ["fracture", "deep_burn"]
    moderate_cases = ["burn", "cut", "laceration"]
    mild_cases = ["bruise", "abrasion", "normal skin"]

    if injury in severe_cases:
        return "شديدة"

    elif injury in moderate_cases:
        if confidence > 0.85:
            return "متوسطة"
        else:
            return "بسيطة"

    elif injury in mild_cases:
        return "بسيطة"

    else:
        return "غير محددة"


# =========================
# 📌 3. CHATBOT PREDICTION
# =========================
def chatbot_predict(img_path):

    # قراءة الصورة
    img = cv2.imread(img_path)
    if img is None:
        return "❌ الصورة غير موجودة أو المسار غير صحيح"

    # تجهيز الصورة
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))

    img_array = np.expand_dims(img, axis=0)
    img_preprocessed = tf.keras.applications.efficientnet.preprocess_input(
        img_array.astype('float32')
    )

    # استخراج features
    features = extractor.predict(img_preprocessed, verbose=0)

    # التوقع باستخدام Ensemble
    probabilities = ensemble.predict_proba(features)[0]
    best_idx = np.argmax(probabilities)

    injury = class_labels[best_idx]
    confidence = probabilities[best_idx]

    # تحديد الشدة
    severity = estimate_severity(injury, confidence)

    # =========================
    # 📌 4. CHATBOT RESPONSE
    # =========================
    response = f"تم التعرف على الإصابة: {injury}\nدرجة الخطورة: {severity}"

    return response


# =========================
# 📌 5. TEST
# =========================
image_path = '/content/drive/MyDrive/images.jpg'

result = chatbot_predict(image_path)
print(result)

تم التعرف على الإصابة: burn
درجة الخطورة: متوسطة


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
import numpy as np
import cv2
import tensorflow as tf

# =========================
# 📌 1. CLASS NAMES
# =========================
class_labels = list(train_gen.class_indices.keys())


# =========================
# 📌 2. SEVERITY LOGIC
# =========================
def estimate_severity(injury, confidence):

    severe_cases = ["fracture", "deep_burn"]
    moderate_cases = ["burn", "cut", "laceration"]
    mild_cases = ["bruise", "abrasion", "normal skin"]

    if injury in severe_cases:
        return "شديدة"

    elif injury in moderate_cases:
        if confidence > 0.85:
            return "متوسطة"
        else:
            return "بسيطة"

    elif injury in mild_cases:
        return "بسيطة"

    else:
        return "غير محددة"


# =========================
# 📌 3. CHATBOT PREDICTION
# =========================
def chatbot_predict(img_path):

    # قراءة الصورة
    img = cv2.imread(img_path)
    if img is None:
        return "❌ الصورة غير موجودة أو المسار غير صحيح"

    # تجهيز الصورة
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))

    img_array = np.expand_dims(img, axis=0)
    img_preprocessed = tf.keras.applications.efficientnet.preprocess_input(
        img_array.astype('float32')
    )

    # استخراج features
    features = extractor.predict(img_preprocessed, verbose=0)

    # التوقع باستخدام Ensemble
    probabilities = ensemble.predict_proba(features)[0]
    best_idx = np.argmax(probabilities)

    injury = class_labels[best_idx]
    confidence = probabilities[best_idx]

    # تحديد الشدة
    severity = estimate_severity(injury, confidence)

    # =========================
    # 📌 4. CHATBOT RESPONSE
    # =========================
    response = f"تم التعرف على الإصابة: {injury}\nدرجة الخطورة: {severity}"

    return response


# =========================
# 📌 5. TEST
# =========================
image_path = '/content/drive/MyDrive/mm.webp'
result = chatbot_predict(image_path)
print(result)

تم التعرف على الإصابة: laceration
درجة الخطورة: متوسطة


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
import numpy as np
import cv2
import tensorflow as tf

# =========================
# 📌 1. CLASS NAMES
# =========================
class_labels = list(train_gen.class_indices.keys())


# =========================
# 📌 2. SEVERITY LOGIC
# =========================
def estimate_severity(injury, confidence):

    severe_cases = ["fracture", "deep_burn"]
    moderate_cases = ["burn", "cut", "laceration"]
    mild_cases = ["bruise", "abrasion", "normal skin"]

    if injury in severe_cases:
        return "شديدة"

    elif injury in moderate_cases:
        if confidence > 0.85:
            return "متوسطة"
        else:
            return "بسيطة"

    elif injury in mild_cases:
        return "بسيطة"

    else:
        return "غير محددة"


# =========================
# 📌 3. CHATBOT PREDICTION
# =========================
def chatbot_predict(img_path):

    # قراءة الصورة
    img = cv2.imread(img_path)
    if img is None:
        return "❌ الصورة غير موجودة أو المسار غير صحيح"

    # تجهيز الصورة
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))

    img_array = np.expand_dims(img, axis=0)
    img_preprocessed = tf.keras.applications.efficientnet.preprocess_input(
        img_array.astype('float32')
    )

    # استخراج features
    features = extractor.predict(img_preprocessed, verbose=0)

    # التوقع باستخدام Ensemble
    probabilities = ensemble.predict_proba(features)[0]
    best_idx = np.argmax(probabilities)

    injury = class_labels[best_idx]
    confidence = probabilities[best_idx]

    # تحديد الشدة
    severity = estimate_severity(injury, confidence)

    # =========================
    # 📌 4. CHATBOT RESPONSE
    # =========================
    response = f"تم التعرف على الإصابة: {injury}\nدرجة الخطورة: {severity}"

    return response


# =========================
# 📌 5. TEST
# =========================
image_path = '/content/drive/MyDrive/ll.jpg'
result = chatbot_predict(image_path)
print(result)

تم التعرف على الإصابة: bruise
درجة الخطورة: بسيطة


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
import numpy as np
import cv2
import tensorflow as tf

# =========================
# 📌 1. CLASS NAMES
# =========================
class_labels = list(train_gen.class_indices.keys())


# =========================
# 📌 2. SEVERITY LOGIC
# =========================
def estimate_severity(injury, confidence):

    severe_cases = ["fracture", "deep_burn"]
    moderate_cases = ["burn", "cut", "laceration"]
    mild_cases = ["bruise", "abrasion", "normal skin"]

    if injury in severe_cases:
        return "شديدة"

    elif injury in moderate_cases:
        if confidence > 0.85:
            return "متوسطة"
        else:
            return "بسيطة"

    elif injury in mild_cases:
        return "بسيطة"

    else:
        return "غير محددة"


# =========================
# 📌 3. CHATBOT PREDICTION
# =========================
def chatbot_predict(img_path):

    # قراءة الصورة
    img = cv2.imread(img_path)
    if img is None:
        return "❌ الصورة غير موجودة أو المسار غير صحيح"

    # تجهيز الصورة
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))

    img_array = np.expand_dims(img, axis=0)
    img_preprocessed = tf.keras.applications.efficientnet.preprocess_input(
        img_array.astype('float32')
    )

    # استخراج features
    features = extractor.predict(img_preprocessed, verbose=0)

    # التوقع باستخدام Ensemble
    probabilities = ensemble.predict_proba(features)[0]
    best_idx = np.argmax(probabilities)

    injury = class_labels[best_idx]
    confidence = probabilities[best_idx]

    # تحديد الشدة
    severity = estimate_severity(injury, confidence)

    # =========================
    # 📌 4. CHATBOT RESPONSE
    # =========================
    response = f"تم التعرف على الإصابة: {injury}\nدرجة الخطورة: {severity}"

    return response


# =========================
# 📌 5. TEST
# =========================
image_path = '/content/drive/MyDrive/mm.webp'
result = chatbot_predict(image_path)
print(result)

تم التعرف على الإصابة: laceration
درجة الخطورة: متوسطة


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# =========================
# 🔗 1. ربط Google Drive
# =========================
from google.colab import drive
drive.mount('/content/drive')

# =========================
# 📦 2. استيراد المكتبات
# =========================
import tensorflow as tf
import joblib
import json

# =========================
# 🧠 3. تحميل الموديلات
# =========================
extractor = tf.keras.models.load_model(
    "/content/drive/MyDrive/wound_extractor_model.h5"
)

ensemble = joblib.load(
    "/content/drive/MyDrive/wound_ensemble_model.pkl"
)

with open(
    "/content/drive/MyDrive/class_indices.json", "r"
) as f:
    class_indices = json.load(f)

# =========================
# ✅ 4. تأكيد التشغيل
# =========================
print("✅ extractor:", type(extractor))
print("✅ ensemble:", type(ensemble))
print("✅ classes:", class_indices)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


✅ extractor: <class 'keras.src.models.functional.Functional'>
✅ ensemble: <class 'sklearn.ensemble._voting.VotingClassifier'>
✅ classes: {'abrasion': 0, 'bruise': 1, 'burn': 2, 'cut': 3, 'laceration': 4, 'normal skin': 5}


In [ ]:
# تحميل الملفات من Colab للجهاز مباشرة
files.download('wound_extractor_model.h5')
files.download('wound_ensemble_model.pkl')
files.download('class_indices.json')

print("🚀 جاري تحويل الملفات لجهازك... راقبي شريط التحميل في المتصفح.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🚀 جاري تحويل الملفات لجهازك... راقبي شريط التحميل في المتصفح.


In [ ]:
!pip list

Package                                  Version
---------------------------------------- ------------------
absl-py                                  1.4.0
accelerate                               1.13.0
access                                   1.1.10.post3
affine                                   2.4.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.5
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.18.4
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
antlr4-python3-runtime                   4.9.3
anyio                          